In [1]:
import redis
import openai
import hashlib
import json
import time
import os
import numpy as np
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

## LLM
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),          # deepseek-v4-flash
    api_key=os.getenv("EVREN_API_KEY"),     
    base_url=os.getenv("EVREN_BASE_URL")
        # https://evren-llmapi.ssyz.org.tr/v1
)   


## llm Test
response = llm.invoke([HumanMessage(content="Who is Atatürk?!")])
print("LLM Response:", response.content)

r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    decode_responses=True
)

r.ping()
print("Ready!")

LLM Response: Mustafa Kemal Atatürk (1881–1938) was a Turkish military officer, revolutionary statesman, and the founder of the Republic of Turkey. He served as Turkey’s first President from 1923 until his death in 1938.

He is best known for:

- Leading Turkey’s War of Independence (1919–1923) after the collapse of the Ottoman Empire.
- Abolishing the Ottoman Sultanate and Caliphate.
- Establishing a modern, secular, and democratic Turkish republic.
- Introducing sweeping political, legal, cultural, and social reforms — including the adoption of the Latin alphabet, women’s suffrage, secular education, and Western-style legal codes.

The surname “Atatürk,” meaning “Father of the Turks,” was given to him in 1934. He remains a highly revered figure in Turkey, and his image and ideas are central to modern Turkish national identity.
Ready!


In [2]:
# ─────────────────────────────────────────────────
# Step 1: Get embeddings using HuggingFace
# all-MiniLM-L6-v2 is lightweight, fast, and local
# It produces a 384-dimensional vector
# ─────────────────────────────────────────────────

from sentence_transformers import SentenceTransformer


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text: str) -> list:
    """Convert text to a 384-dimensional embedding vector."""
    # encode() numpy array döner, listeye çeviriyoruz
    embedding = embedding_model.encode(text.strip())
    return embedding.tolist()

# Let's see what an embedding looks like
sample_embedding = get_embedding("What is machine learning?")
print(f"Embedding for 'What is machine learning?'")
print(f"  Type          : {type(sample_embedding)}")
print(f"  Dimensions    : {len(sample_embedding)}")
print(f"  First 5 values: {[round(x, 4) for x in sample_embedding[:5]]}")
print(f"  Last 5 values : {[round(x, 4) for x in sample_embedding[-5:]]}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding for 'What is machine learning?'
  Type          : <class 'list'>
  Dimensions    : 384
  First 5 values: [-0.02, 0.0099, 0.0102, 0.0296, 0.0272]
  Last 5 values : [0.0593, 0.0286, 0.0611, 0.067, -0.0302]


In [4]:
# ─────────────────────────────────────────────────
# Step 2: Compute cosine similarity between two vectors
# ─────────────────────────────────────────────────

def cosine_similarity(a: list, b: list) -> float:
    """Cosine similarity between two embedding vectors. Returns 0.0 to 1.0."""
    a_arr = np.array(a)
    b_arr = np.array(b)
    dot_product = np.dot(a_arr, b_arr)
    magnitude_a = np.linalg.norm(a_arr)
    magnitude_b = np.linalg.norm(b_arr)
    return float(dot_product / (magnitude_a * magnitude_b))

# Demo: Compare several texts
texts = [
    "What is machine learning?",
    "Can you explain machine learning to me?",
    "Explain ML",
    "Define machine learning",
    "What is deep learning?",   # related but different
    "How do I make pasta?",     # completely unrelated
]

print("\n" + "═" * 70)
print("\033[1;36m  📐  COSINE SIMILARITY DEMO\033[0m")
print("\033[90m  Comparing semantic similarity of text embeddings\033[0m")
print("═" * 70)

print(f"\n  \033[1;33m⚙️  Computing embeddings for {len(texts)} texts...\033[0m")
embeddings = {text: get_embedding(text) for text in texts}
print(f"  \033[1;32m✅ Done.\033[0m")

reference = "What is machine learning?"
ref_embedding = embeddings[reference]

print("\n" + "─" * 70)
print(f"\033[1;34m  🎯  Reference:\033[0m \033[1;37m'{reference}'\033[0m")
print("─" * 70)

# ─────────────────────────────────────────────
# COMPUTE AND DISPLAY SIMILARITIES
# ─────────────────────────────────────────────
results = []
for text, emb in embeddings.items():
    if text == reference:
        continue
    sim = cosine_similarity(ref_embedding, emb)
    results.append((sim, text))

# Sort descending by similarity
results.sort(reverse=True, key=lambda x: x[0])

for sim, text in results:
    # Determine color and label based on similarity score
    if sim >= 0.85:
        color = "\033[1;32m"      # green = very similar
        label = "HIGH"
    elif sim >= 0.70:
        color = "\033[1;36m"      # cyan = similar
        label = "MED "
    elif sim >= 0.50:
        color = "\033[1;33m"      # yellow = somewhat related
        label = "LOW "
    else:
        color = "\033[1;31m"      # red = unrelated
        label = "NONE"

    # Build a visual bar (0-40 chars)
    bar_len = int(max(sim, 0) * 40)
    bar = "█" * bar_len + "\033[90m" + "░" * (40 - bar_len) + "\033[0m"

    print(f"\n  {color}[{label}]\033[0m  {color}{sim:.3f}\033[0m  {bar}")
    print(f"           \033[90m'{text}'\033[0m")

print("\n" + "═" * 70)
print(f"\033[90m  💡 Interpretation:\033[0m")
print(f"     \033[1;32m█ HIGH\033[0m   ≥ 0.85   \033[90m→ nearly identical meaning\033[0m")
print(f"     \033[1;36m█ MED \033[0m   ≥ 0.70   \033[90m→ closely related\033[0m")
print(f"     \033[1;33m█ LOW \033[0m   ≥ 0.50   \033[90m→ somewhat related\033[0m")
print(f"     \033[1;31m█ NONE\033[0m   < 0.50   \033[90m→ unrelated\033[0m")
print("═" * 70 + "\n")


══════════════════════════════════════════════════════════════════════
  📐  COSINE SIMILARITY DEMO
  Comparing semantic similarity of text embeddings
══════════════════════════════════════════════════════════════════════

  ⚙️  Computing embeddings for 6 texts...
  ✅ Done.

──────────────────────────────────────────────────────────────────────
  🎯  Reference: 'What is machine learning?'
──────────────────────────────────────────────────────────────────────

  [HIGH]  0.911  ████████████████████████████████████░░░░
           'Define machine learning'

  [MED ]  0.824  ████████████████████████████████░░░░░░░░
           'Can you explain machine learning to me?'

  [LOW ]  0.683  ███████████████████████████░░░░░░░░░░░░░
           'What is deep learning?'

  [NONE]  0.476  ███████████████████░░░░░░░░░░░░░░░░░░░░░
           'Explain ML'

  [NONE]  0.098  ███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
           'How do I make pasta?'

══════════════════════════════════════════════════════════

## Thresholds

In [5]:
# Demo: how the threshold affects what counts as a cache hit
test_pairs = [
    ("What is machine learning?", "Can you explain machine learning?"),
    ("What is machine learning?", "Explain ML to me"),
    ("What is machine learning?", "What is deep learning?"),
    ("What is machine learning?", "How do I make pasta?"),
]

thresholds = [0.65, 0.70, 0.75, 0.80]

print("\n" + "═" * 96)
print("\033[1;36m  🎚️   THRESHOLD SENSITIVITY DEMO\033[0m")
print("\033[90m  How the similarity threshold affects cache hit decisions\033[0m")
print("═" * 96)

# ─────────────────────────────────────────────
# TABLE HEADER
# ─────────────────────────────────────────────
header = f"  \033[1;33m{'TEXT PAIR':<58} │ {'SIM':>5} │\033[0m"
print(header, end="")
for t in thresholds:
    print(f" \033[1;33m{t:>5}\033[0m", end="")
print()

print("  " + "─" * 92)

# ─────────────────────────────────────────────
# COMPUTE AND DISPLAY EACH PAIR
# ─────────────────────────────────────────────
for t1, t2 in test_pairs:
    e1 = get_embedding(t1)
    e2 = get_embedding(t2)
    sim = cosine_similarity(e1, e2)

    # Truncate long text for display
    label = f"'{t2[:53]}'" if len(t2) > 55 else f"'{t2}'"

    # Determine color for the similarity score
    if sim >= 0.85:
        sim_color = "\033[1;32m"     # green
    elif sim >= 0.70:
        sim_color = "\033[1;36m"     # cyan
    elif sim >= 0.50:
        sim_color = "\033[1;33m"     # yellow
    else:
        sim_color = "\033[1;31m"     # red

    print(f"  \033[90m{label:<58}\033[0m │ {sim_color}{sim:>5.3f}\033[0m │", end="")

    # Print HIT / miss for each threshold with color
    for t in thresholds:
        if sim >= t:
            print(f" \033[1;32m{'✓ HIT':>5}\033[0m", end="")
        else:
            print(f" \033[90m{'· miss':>5}\033[0m", end="")
    print()

print("  " + "─" * 92)

# ─────────────────────────────────────────────
# SUMMARY — HIT COUNT PER THRESHOLD
# ─────────────────────────────────────────────
print(f"\n  \033[1;34m📊 Hit count per threshold:\033[0m")
print(f"  \033[90m{'THRESHOLD':<12} {'HITS':>6}  {'MISSES':>7}  {'HIT RATE':>10}\033[0m")
print("  " + "─" * 42)

for t in thresholds:
    hits = sum(1 for t1, t2 in test_pairs
               if cosine_similarity(get_embedding(t1), get_embedding(t2)) >= t)
    misses = len(test_pairs) - hits
    hit_rate = hits / len(test_pairs) * 100

    # Determine color based on hit rate
    if hit_rate >= 75:
        rate_color = "\033[1;32m"     # green
    elif hit_rate >= 50:
        rate_color = "\033[1;33m"     # yellow
    else:
        rate_color = "\033[1;31m"     # red

    print(f"  \033[1;37m{t:<12}\033[0m \033[1;32m{hits:>6}\033[0m  "
          f"\033[90m{misses:>7}\033[0m  {rate_color}{hit_rate:>9.0f}%\033[0m")

print("  " + "─" * 42)
print("\n" + "═" * 96)

print(f"\n  \033[90m💡 Insight: Lower thresholds catch more semantic matches\033[0m")
print(f"  \033[90m   but risk false positives. Higher thresholds are safer but\033[0m")
print(f"  \033[90m   miss legitimate paraphrases.\033[0m\n")


════════════════════════════════════════════════════════════════════════════════════════════════
  🎚️   THRESHOLD SENSITIVITY DEMO
  How the similarity threshold affects cache hit decisions
════════════════════════════════════════════════════════════════════════════════════════════════
  TEXT PAIR                                                  │   SIM │  0.65   0.7  0.75   0.8
  ────────────────────────────────────────────────────────────────────────────────────────────
  'Can you explain machine learning?'                        │ 0.847 │ ✓ HIT ✓ HIT ✓ HIT ✓ HIT
  'Explain ML to me'                                         │ 0.484 │ · miss · miss · miss · miss
  'What is deep learning?'                                   │ 0.683 │ ✓ HIT · miss · miss · miss
  'How do I make pasta?'                                     │ 0.098 │ · miss · miss · miss · miss
  ────────────────────────────────────────────────────────────────────────────────────────────

  📊 Hit count per threshold:
  THRE

## Semantic Caching

In [6]:
import json
import time
import hashlib
import numpy as np
from langchain_core.messages import HumanMessage

# ─────────────────────────────────────────────
# CONFIGURATION & CONSTANTS
# ─────────────────────────────────────────────
SEMANTIC_CACHE_PREFIX = "sem:cache:"
SEMANTIC_INDEX_KEY   = "sem:cache:index"  # Set of all cache entry IDs
SIMILARITY_THRESHOLD = 0.60

# Cost constants (Evren API / DeepSeek)
COST_INPUT  = 0.00000015
COST_OUTPUT = 0.00000060

def cosine_similarity(v1: list, v2: list) -> float:
    """Compute cosine similarity between two vectors."""
    a = np.array(v1)
    b = np.array(v2)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm == 0:
        return 0.0
    return float(np.dot(a, b) / norm)


def store_in_semantic_cache(prompt: str, response: str, embedding: list):
    """Store a prompt-response pair along with its embedding in Redis."""
    entry_id = hashlib.md5(prompt.encode("utf-8")).hexdigest()
    key = f"{SEMANTIC_CACHE_PREFIX}{entry_id}"

    # Hashes store strings; serialize embedding with json.dumps
    r.hset(key, mapping={
        "prompt": prompt,
        "response": response,
        "embedding": json.dumps(embedding),
        "stored_at": str(time.time())
    })
    r.sadd(SEMANTIC_INDEX_KEY, entry_id)  # Track all entry IDs in a Set
    return entry_id


def find_semantic_match(query_embedding: list, threshold: float = SIMILARITY_THRESHOLD):
    """Search all cached embeddings and return the best match if above threshold."""
    entry_ids = r.smembers(SEMANTIC_INDEX_KEY)
    if not entry_ids:
        return None, 0.0

    best_score = 0.0
    best_entry = None

    for entry_id in entry_ids:
        data = r.hgetall(f"{SEMANTIC_CACHE_PREFIX}{entry_id}")
        if not data or "embedding" not in data:
            continue
        cached_embedding = json.loads(data["embedding"])
        score = cosine_similarity(query_embedding, cached_embedding)
        if score > best_score:
            best_score = score
            best_entry = data

    if best_score >= threshold:
        return best_entry, best_score
    return None, best_score


def ask_semantic(prompt: str, threshold: float = SIMILARITY_THRESHOLD) -> dict:
    """Ask LLM with semantic caching."""
    start = time.time()

    # Step 1: Compute embedding for the incoming question
    query_embedding = get_embedding(prompt)

    # Step 2: Search cache
    match, score = find_semantic_match(query_embedding, threshold)

    if match:
        elapsed_ms = (time.time() - start) * 1000
        return {
            "response": match["response"],
            "source": "SEMANTIC CACHE HIT",
            "similarity": round(score, 4),
            "matched_prompt": match["prompt"],
            "time_ms": round(elapsed_ms, 2),
            "cost_usd": 0.0,
            "input_tokens": 0,
            "output_tokens": 0,
        }

    # Step 3: Cache miss — call your LLM (LangChain ChatOpenAI instance)
    ai_msg = llm.invoke([HumanMessage(content=prompt)])
    response_text = ai_msg.content

    # Extract token usage from LangChain message metadata
    usage = getattr(ai_msg, "usage_metadata", {}) or {}
    input_tokens = usage.get("input_tokens", 0)
    output_tokens = usage.get("output_tokens", 0)
    cost = (input_tokens * COST_INPUT) + (output_tokens * COST_OUTPUT)

    # Step 4: Store in semantic cache
    store_in_semantic_cache(prompt, response_text, query_embedding)

    elapsed_ms = (time.time() - start) * 1000
    return {
        "response": response_text,
        "source": f"API CALL (best similarity: {score:.3f})",
        "similarity": round(score, 4),
        "matched_prompt": None,
        "time_ms": round(elapsed_ms, 2),
        "cost_usd": round(cost, 8),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }


# ─────────────────────────────────────────────
# OUTPUT
# ─────────────────────────────────────────────
print("\n" + "═" * 62)
print("\033[1;36m  🧠  SEMANTIC CACHE MODULE\033[0m")
print("\033[90m  Embedding-based cache with cosine similarity matching\033[0m")
print("═" * 62)

print(f"\n  \033[1;33m▸ Prefix\033[0m            → \033[1;37m{SEMANTIC_CACHE_PREFIX}\033[0m")
print(f"  \033[1;33m▸ Index Key\033[0m         → \033[1;37m{SEMANTIC_INDEX_KEY}\033[0m")
print(f"  \033[1;33m▸ Threshold\033[0m         → \033[1;32m{SIMILARITY_THRESHOLD}\033[0m")
print(f"  \033[1;33m▸ Cost (input/token)\033[0m → \033[90m${COST_INPUT}\033[0m")
print(f"  \033[1;33m▸ Cost (output/token)\033[0m→ \033[90m${COST_OUTPUT}\033[0m")

print(f"\n  \033[1;32m✅ Functions defined:\033[0m")
print(f"     \033[90m• cosine_similarity()\033[0m")
print(f"     \033[90m• store_in_semantic_cache()\033[0m")
print(f"     \033[90m• find_semantic_match()\033[0m")
print(f"     \033[90m• ask_semantic()\033[0m")

print("═" * 62 + "\n")



══════════════════════════════════════════════════════════════
  🧠  SEMANTIC CACHE MODULE
  Embedding-based cache with cosine similarity matching
══════════════════════════════════════════════════════════════

  ▸ Prefix            → sem:cache:
  ▸ Index Key         → sem:cache:index
  ▸ Threshold         → 0.6
  ▸ Cost (input/token) → $1.5e-07
  ▸ Cost (output/token)→ $6e-07

  ✅ Functions defined:
     • cosine_similarity()
     • store_in_semantic_cache()
     • find_semantic_match()
     • ask_semantic()
══════════════════════════════════════════════════════════════



In [7]:
# Clear any existing semantic cache
for key in r.keys("sem:cache:*"):
    r.delete(key)
print("Cleared old semantic cache entries")

Cleared old semantic cache entries


In [8]:
# First call — original prompt (will call API and store in cache)
print("\n" + "═" * 62)
print("\033[1;36m  🌐  CALL 1  \033[0m\033[90m— Original question (expected: API CALL)\033[0m")
print("═" * 62)

result = ask_semantic("What is machine learning?")

# Determine color, icon and label based on source
source = result['source']
if "SEMANTIC CACHE HIT" in source:
    source_display = "\033[1;32m⚡ SEMANTIC CACHE HIT\033[0m"
elif "API CALL" in source:
    source_display = f"\033[1;33m🌐 {source}\033[0m"
else:
    source_display = f"\033[1;37m{source}\033[0m"

# Color the similarity score based on value
sim = result['similarity']
if sim >= 0.85:
    sim_color = "\033[1;32m"
elif sim >= 0.70:
    sim_color = "\033[1;36m"
elif sim >= 0.50:
    sim_color = "\033[1;33m"
else:
    sim_color = "\033[1;31m"

print(f"\n  \033[1;34m▸ Source\033[0m      → {source_display}")
print(f"  \033[1;34m▸ Similarity\033[0m  → {sim_color}{sim:.4f}\033[0m")
print(f"  \033[1;34m▸ Time\033[0m        → \033[1;32m{result['time_ms']} ms\033[0m")
print(f"  \033[1;34m▸ Cost\033[0m        → \033[1;33m${result['cost_usd']}\033[0m")

print(f"\n  \033[1;34m▸ Response\033[0m")
print(f"     \033[90m{result['response'][:200]}...\033[0m")

print("\n" + "═" * 62 + "\n")


══════════════════════════════════════════════════════════════
  🌐  CALL 1  — Original question (expected: API CALL)
══════════════════════════════════════════════════════════════

  ▸ Source      → 🌐 API CALL (best similarity: 0.000)
  ▸ Similarity  → 0.0000
  ▸ Time        → 3056.52 ms
  ▸ Cost        → $0.0002088

  ▸ Response
     Machine learning is a branch of artificial intelligence (AI) that gives computers the ability to learn from data and improve their performance over time without being explicitly programmed for every t...

══════════════════════════════════════════════════════════════



In [9]:
# Now try semantically similar questions
# These should all hit the semantic cache!

similar_questions = [
    "Can you explain machine learning to me?",
    "What does machine learning mean?",
    "Define machine learning",
    "Explain Machine learning",
    "What is deep learning?",           # related but different concept
    "How do I bake a chocolate cake?",  # completely unrelated
]

print("\n" + "═" * 100)
print("\033[1;36m  🔍  SEMANTIC CACHE LOOKUP TEST\033[0m")
print(f"\033[90m  {len(similar_questions)} similar questions · expecting mostly CACHE HITs\033[0m")
print("═" * 100)

# ─────────────────────────────────────────────
# TABLE HEADER
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m{'#':>2}  {'SOURCE':<16} {'SIM':>6}  {'TIME (ms)':>10}  QUESTION\033[0m")
print("  " + "─" * 96)

# ─────────────────────────────────────────────
# PROCESS EACH QUESTION
# ─────────────────────────────────────────────
cache_hits = 0
api_calls = 0

for i, q in enumerate(similar_questions, 2):
    result = ask_semantic(q)

    # Determine color, icon and label based on source
    if "CACHE HIT" in result["source"]:
        src_label = "\033[1;32m⚡ CACHE HIT\033[0m"
        cache_hits += 1
    else:
        src_label = "\033[1;33m🌐 API CALL\033[0m"
        api_calls += 1

    # Color the similarity score
    sim = result['similarity']
    if sim >= 0.85:
        sim_color = "\033[1;32m"
    elif sim >= 0.70:
        sim_color = "\033[1;36m"
    elif sim >= 0.50:
        sim_color = "\033[1;33m"
    else:
        sim_color = "\033[1;31m"

    print(f"  \033[90m{i:>2}\033[0m  {src_label:<24} "
          f"{sim_color}{sim:>6.3f}\033[0m  "
          f"\033[1;32m{result['time_ms']:>8.1f}\033[0m  "
          f"\033[90m{q[:52]}\033[0m")

    # Show matched prompt if it exists
    if result["matched_prompt"]:
        matched_preview = result["matched_prompt"][:60]
        print(f"      \033[90m↳ matched:\033[0m \033[1;37m'{matched_preview}'\033[0m")

print("  " + "─" * 96)

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
total = len(similar_questions)
hit_rate = cache_hits / total * 100

print(f"\n  \033[1;34m📊 Summary:\033[0m")
print(f"     \033[1;32m⚡ Cache hits\033[0m   → \033[1;32m{cache_hits:>2}\033[0m")
print(f"     \033[1;33m🌐 API calls\033[0m    → \033[1;33m{api_calls:>2}\033[0m")
print(f"     \033[1;34m📈 Hit rate\033[0m     → \033[1;37m{hit_rate:.0f}%\033[0m")

print("\n" + "═" * 100 + "\n")


════════════════════════════════════════════════════════════════════════════════════════════════════
  🔍  SEMANTIC CACHE LOOKUP TEST
  6 similar questions · expecting mostly CACHE HITs
════════════════════════════════════════════════════════════════════════════════════════════════════

   #  SOURCE              SIM   TIME (ms)  QUESTION
  ────────────────────────────────────────────────────────────────────────────────────────────────
   2  ⚡ CACHE HIT    0.824      57.7  Can you explain machine learning to me?
      ↳ matched: 'What is machine learning?'
   3  ⚡ CACHE HIT    0.944      27.4  What does machine learning mean?
      ↳ matched: 'What is machine learning?'
   4  ⚡ CACHE HIT    0.911      28.9  Define machine learning
      ↳ matched: 'What is machine learning?'
   5  ⚡ CACHE HIT    0.841      19.1  Explain Machine learning
      ↳ matched: 'What is machine learning?'
   6  ⚡ CACHE HIT    0.683      22.0  What is deep learning?
      ↳ matched: 'What is machine learning?'
 

In [10]:
# How many entries are in the semantic cache now?
index_size = r.scard(SEMANTIC_INDEX_KEY)

print("\n" + "═" * 70)
print("\033[1;36m  🗂️   SEMANTIC CACHE INVENTORY\033[0m")
print("═" * 70)

print(f"\n  \033[1;34m▸ Total unique entries\033[0m → \033[1;37m{index_size}\033[0m")

# ─────────────────────────────────────────────
# LIST ALL CACHED ENTRIES
# ─────────────────────────────────────────────
entry_ids = r.smembers(SEMANTIC_INDEX_KEY)

if not entry_ids:
    print(f"\n  \033[1;33m⚠️  The semantic cache is currently empty.\033[0m\n")
else:
    print(f"\n  \033[1;33m📋 Cached prompts:\033[0m\n")

    for idx, entry_id in enumerate(entry_ids, 1):
        data = r.hgetall(f"{SEMANTIC_CACHE_PREFIX}{entry_id}")

        # Skip malformed entries (missing prompt field)
        if not data or "prompt" not in data:
            print(f"  \033[90m[{idx:>2}] ⚠️  (malformed entry: {entry_id})\033[0m")
            continue

        prompt = data['prompt']

        # Shorten entry_id for display (show prefix + first 8 chars)
        short_id = f"{entry_id[:8]}…" if len(entry_id) > 8 else entry_id

        print(f"  \033[1;34m[{idx:>2}]\033[0m \033[90m{short_id}\033[0m  \033[1;37m'{prompt}'\033[0m")

print("\n" + "═" * 70 + "\n")


══════════════════════════════════════════════════════════════════════
  🗂️   SEMANTIC CACHE INVENTORY
══════════════════════════════════════════════════════════════════════

  ▸ Total unique entries → 2

  📋 Cached prompts:

  [ 1] 3adda8c1…  'What is machine learning?'
  [ 2] ed2fc87e…  'How do I bake a chocolate cake?'

══════════════════════════════════════════════════════════════════════

